<h1 style="text-align: center; font-size: 50px;"> Register Model </h1>

# Notebook Overview

- Configure the Environment
- Define Constants and Paths
- Define Necessary Modules
- Define MLflow Model Class
- Log and Register Model
- Prepare Eval Data
- Evaluate Model
- Log Execution Time

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 14.4 ms, sys: 12.3 ms, total: 26.7 ms
Wall time: 1.07 s


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 8


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

## Configure the Environment

In [3]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Local application-specific imports
from src.utils import logger

In [4]:
start_time = time.time()
logger.info("✅ Notebook execution started.")

In [5]:
# Standard & Third-Party Libraries
import os
import sys
import json
import re
import yaml
from pathlib import Path
from collections import defaultdict
import pandas as pd
import mlflow
from mlflow.models import ModelSignature, evaluate
from mlflow.types.schema import Schema, ColSpec
from typing import List
import mlflow
import pandas as pd

# Import standard MLflow metrics
from mlflow.metrics import exact_match, rouge1, rougeL

# Import new Logger for models-from-code
from src.mlflow import Logger
from src.utils import load_config, configure_proxy

from IPython import get_ipython

In [6]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Define Constants and Paths

In [7]:
# Configuration Paths
CONFIG_PATH = Path("../configs/config.yaml")
REQUIREMENTS_PATH = Path("../requirements.txt")
SECRETS_PATH = Path("../configs/secrets.yaml")

# MLflow Configuration
MLFLOW_EXPERIMENT_NAME = "Markdown_Correction_Service"
MLFLOW_MODEL_NAME = "markdown-corrector"

## Define Necessary Modules

In [ ]:
from langchain_core.prompts import PromptTemplate

## Define MLflow Model Class

## Log and Register Model

In [9]:
%%time

# Load configuration
config = load_config(str(CONFIG_PATH))

# Setup proxy
configure_proxy(config)

# Extract model path from config
model_path = config.get("model_path")

# Load secrets
if SECRETS_PATH.exists():
    with open(SECRETS_PATH, "r") as f:
        secrets = yaml.safe_load(f) or {}
else:
    secrets = {}

# Validate model path if provided
if model_path and not os.path.exists(model_path):
    print(f"Warning: Model file not found at {model_path}")
    print("Make sure the model path in config.yaml is correct")

# Define the model's signature
input_schema = Schema([
    ColSpec("string", "repo_url"),
    ColSpec("string", "files")
])
output_schema = Schema([
    ColSpec("string", "corrected"),
    ColSpec("string", "originals"),
    ColSpec("double", "response_time"),
    ColSpec("string", "evaluation_metrics")
])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Set up MLflow experiment
mlflow.set_tracking_uri('/phoenix/mlflow')
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="GrammarCorrector_ModelsFromCode") as run:
    run_id = run.info.run_id
    logger.info(f"Starting MLflow run with ID: {run_id}")
    
    # Use new Logger.log_model approach
    Logger.log_model(
        signature=signature,
        artifact_path=MLFLOW_MODEL_NAME,
        config_path=str(CONFIG_PATH),
        secrets_dict=secrets,
        model_path=model_path,
        demo_folder="../demo"
    )
    logger.info(f"✅ Model '{MLFLOW_MODEL_NAME}' logged successfully using models-from-code approach.")
    
    # Register the model to the model registry
    model_uri = f"runs:/{run_id}/{MLFLOW_MODEL_NAME}"
    registered_model = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
    logger.info(f"✅ Model registered to registry as '{MLFLOW_MODEL_NAME}' (version {registered_model.version})")


/opt/conda/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/16 20:43:04 INFO mlflow.tracking.fluent: Experiment with name 'Markdown_Correction_Service' does not exist. Creating a new experiment.


2026/04/16 20:44:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/opt/conda/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3304: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


/opt/conda/lib/python3.12/site-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'markdown-corrector'.
2026/04/16 20:47:28 WARNING mlflow.tracking._model_registry.fluent: Run with id bcf0202beedc4e5dbbc87762c20d579b has no artifacts at artifact path 'markdown-corrector', registering model based on models:/m-1d24ebbbd215422f8b38f4ce6694e9c6 instead
Created version '1' of model 'markdown-corrector'.


CPU times: user 1.13 s, sys: 23.6 s, total: 24.8 s
Wall time: 4min 24s


## Prepare Eval Data

In [10]:
%%time

logger.info("Preparing data for evaluation…")

# Load the newly registered model
model_uri    = f"models:/{MLFLOW_MODEL_NAME}/latest"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Get original content from the test repo
test_repo_url = "https://github.com/hp-david/grammarbptest"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Load secrets
github_token = os.getenv("AIS_GITHUB_ACCESS_TOKEN")
if github_token:
    secrets = {"AIS_GITHUB_ACCESS_TOKEN": github_token}
    logger.info("Loaded GITHUB_ACCESS_TOKEN from environment variable.")
else:
    try:
        secrets_path = context.artifacts.get("secrets")
    except NameError:
        secrets_path = "../configs/secrets.yaml" # Fallback for standalone execution

    if secrets_path and os.path.exists(secrets_path):
        with open(secrets_path, "r") as f:
            secrets = yaml.safe_load(f)
        logger.info(f"Loaded secrets from {secrets_path}.")
    else:
        secrets = {}
        logger.warning("No GITHUB_ACCESS_TOKEN found in environment or secrets.yaml.")

# Instantiate and fetch raw markdowns
processor = GitHubMarkdownProcessor(
    repo_url=     test_repo_url,
    access_token= secrets.get("AIS_GITHUB_ACCESS_TOKEN") # Use .get() for safety
)
original_markdowns = processor.run()

# Initialize a dictionary to store all corrected results
all_corrected = {}
logger.info(f"Starting prediction for {len(original_markdowns)} files...")

# Loop through each file and call predict, just like the frontend
for filename, content in original_markdowns.items():
    logger.info(f"  - Predicting for: {filename}")
    
    # Create a dataframe with a SINGLE file's content string
    test_input = pd.DataFrame([{"repo_url": None, "files": content}])
    
    # Run prediction for this single file
    prediction_df = loaded_model.predict(test_input)
    
    # Extract the corrected content dictionary from the prediction
    corrected_dict = prediction_df.loc[0, "corrected"]
    
    # Store the corrected text using the original filename as the key
    all_corrected[filename] = corrected_dict.get("corrected_file.md", content)

logger.info("✅ Prediction complete for all files.")

eval_data = []
for fn, orig in original_markdowns.items():
    if fn in all_corrected:
        eval_data.append({
            "original":  orig,
            "corrected": all_corrected[fn]
        })

evaluation_df = pd.DataFrame(eval_data)
logger.info(f"✅ Created evaluation DataFrame with {len(evaluation_df)} file(s).")
display(evaluation_df.head())

/home/jovyan/AI-Blueprints/generative-ai/grammar-correction-with-langchain/src/mlflow/loader.py:71: UserWarning: WARNING! low_vram is not default parameter.
                low_vram was transferred to model_kwargs.
                Please confirm that low_vram is what you intended.
  model = Model(config=config, secrets=secrets, model_path=model_path)
/home/jovyan/AI-Blueprints/generative-ai/grammar-correction-with-langchain/src/mlflow/loader.py:71: UserWarning: WARNING! rope_scaling is not default parameter.
                rope_scaling was transferred to model_kwargs.
                Please confirm that rope_scaling is what you intended.
  model = Model(config=config, secrets=secrets, model_path=model_path)
/home/jovyan/AI-Blueprints/generative-ai/grammar-correction-with-langchain/src/mlflow/loader.py:71: UserWarning: WARNING! num_threads is not default parameter.
                num_threads was transferred to model_kwargs.
                Please confirm that num_threads is what you i

,original,corrected
0,"<h1 style=""text-align: center; font-size: 40px...","<h1 style=""text-align: center; font-size: 40px..."
1,# 📜💬 Shakespeare text generation with RNN\n\n<...,# 📜💬 Shakespeare text generation with RNN\n\n<...


CPU times: user 2min 47s, sys: 31 s, total: 3min 17s
Wall time: 4min 48s


## Log Evaluation Metrics

In [11]:
%%time

with mlflow.start_run(run_id=run_id):
    logger.info(f"Logging metrics for run ID: {run_id}")

    metrics_to_log = prediction_df.loc[0, "evaluation_metrics"]

    # Log the dictionary of metrics to MLflow
    mlflow.log_metrics(metrics_to_log)

    logger.info("✅ Metrics logged successfully.")
    logger.info(json.dumps(metrics_to_log, indent=2))

CPU times: user 0 ns, sys: 30 ms, total: 30 ms
Wall time: 509 ms


## Log Execution Time

In [12]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).